---
title: "fast.ai book questionnaire 2"
author: "Harish"
date: "2024-04-11"
categories: [Deep Learning, fastai, quiz]
# image: thumbnail.jpg
---

1. Provide an example of where the bear classification model might work poorly in production, due to structural or style differences in the training data.
2. Where do text models currently have a major deficiency?
3. What are possible negative societal implications of text generation models?
4. In situations where a model might make mistakes, and those mistakes could be harmful, what is a good alternative to automating a process?
5. What kind of tabular data is deep learning particularly good at?
6. What's a key downside of directly using a deep learning model for recommendation systems?
7. What are the steps of the Drivetrain Approach?
8. How do the steps of the Drivetrain Approach map to a recommendation system?


12. What does the splitter parameter to DataBlock do?
13. How do we ensure a random split always gives the same validation set?
14. What letters are often used to signify the independent and dependent variables?
15. What's the difference between the crop, pad, and squish resize approaches? When might you choose one over the others?
16. What is data augmentation? Why is it needed?
17. What is the difference between item_tfms and batch_tfms?
18. What is a confusion matrix?
19. What does export save?
20. What is it called when we use a model for getting predictions, instead of training?
21. What are IPython widgets?
22. When might you want to use CPU for deployment? When might GPU be better?
23. What are the downsides of deploying your app to a server, instead of to a client (or edge) device such as a phone or PC?
24. What are three examples of problems that could occur when rolling out a bear warning system in practice?
25. What is "out-of-domain data"?
26. What is "domain shift"?
27. What are the three steps in the deployment process?

#### 9. Create an image recognition model using data you curate, and deploy it on the web.

Future Post

#### 10. What is DataLoaders?


`DataLoaders` is a class provided by *fastai* to store collection of `DataLoader` objects passed to it. The first two passed are available as `train` and `valid`.

```python
dls = DataLoaders(...) # assume few DataLoader objects are passed as input.
train = dls.train      # accessing training DataLoader
valid = dls.valid      # accessing validation DataLoader
```

For problem specific datasets, *fastai* also provides factory methods like `ImageDataLoaders`, `TextDataLoaders`, `TabularDataLoaders`, `CollabDataLoaders`, `SegmentationDataLoaders`.

::: {.callout-note}
***factory methods*** - a class that has one creation method with a large conditional that based on method parameters chooses which product class to instantiate and then return. *Source: https://refactoring.guru/*

#### 11. What four things do we need to tell fastai to create DataLoaders?

* What kinds of data we are working with?
* How to get the list of items?
* How to label these items?
* How to create a validation set?

#### ⚡ Write code to create a image datalaoder from a classification dataset where each class has its own subfolder?

#### ⚡ What is `ImageDataLoaders` and what are the available factory methods?

It is a Basic wrapper around several DataLoaders with factory methods for computer vision problems. This class should not be used directly, one of the factory methods should be preferred instead.

| factory methods | description | when to use? |
| --------------- | ----------- | ------------ |
| `ImageDataLoaders.from_folder` | Create from imagenet style dataset in `path` with `train` and `valid` subfolders | each label is categorized into subfolders under `train` and `valid` folders |
| `ImageDataLoaders.from_path_func` | Create from list of `fnames` in `path`s with `label_func` | when label can be computed from `path` of the sample. Use function to compute the label. |
| `ImageDataLoaders.from_path_re` | Create from list of `fnames` in `path`s with re expression `pat` | when label is part of the `path` of the sample. Use regex patterns to extract the label. |
| `ImageDataLoaders.from_name_func` | Create from the name attrs of `fnames` in `path`s with `label_func` | when label can be computed from the file name of the sample. Use function to compute the label from filename. |
| `ImageDataLoaders.from_name_re` | Create from the name attrs of `fnames` in `path`s with re expression `pat` | when label can be extracted from the file name of the sample. Use regex patterns. |
| `ImageDataLoaders.from_df` | Create from `df` using `fn_col` and `label_col` | when dataframe has columns specifying the input file path and the corresponding label as columns |
| `ImageDataLoaders.from_csv` | Create from `path/csv_fname` using `fn_col` and `label_col` | same as `from_df` after loading the file |
| `ImageDataLoaders.from_lists` | Create from list of `fnames` and `labels` in path | when we have just the filenames and labels in the path |

All the above factory methods also accept as arguments:
* `item_tfms`: one or several transforms applied to the items before batching them
* `batch_tfms`: one or several transforms applied to the batches once they are formed
* `bs`: the batch size
* `val_bs`: the batch size for the validation DataLoader (defaults to `bs`)
* `shuffle_train`: if we shuffle the training DataLoader or not
* `device`: the PyTorch device to use (defaults to `default_device()`)


#### ⚡ What is a `DataBlock`?
There can be applications and data structures that fit to predefined DataLoaders methods like `ImageDataLoaders`, `TextDataLoaders` etc., When we encounter applications where this is not the case, *fastai* provides `DataBlock` API which is helpful in designing our own DataLoaders.

#### ⚡ Define a `DataBlock` for image classification task?

```python
db = DataBlock( # <1>
    blocks=(ImageBlock, CategoryBlock), # <2>
    get_items=get_image_files, # <3>
    splitter=RandomSplitter(valid_pct=0.2, seed=42), # <4>
    get_y=parent_label, # <5>
    item_tfms=[Resize(192, method='squish')] # <6>
)
```

1. `DataBlock` is the class used to write the blueprint of our dataset.
2. What are the input (independent variable) and output (dependent variable) data types? `ImageBlock` tells the input variable in our dataset represents **Image** datatype and `CategoryBlock` tells the output variable in our dataset represents **Category** datatype (classification task).
3. How do we extract individual items of our dataset? `get_image_files` is a function which takes root folder path as input and returns all the image files as output.
4. How to split the dataset into train and validation sets? `RandomSplitter` is one of the splitting strategies. It splits the dataset randomly. `valid_pct=0.2` tells 20% of the dataset should be part of validation set. `seed=42` is used for reproducibility.
5. How to create label (`y`) for each input sample in our dataset? `parent_label` is a function which extracts the parent folder name of the file and returns as output.
6. Any transformations to be applied on each item of the dataset? Usually when we train the model, we won't be training one image at a time but a batch of images. To make this convenient, we make sure all the images are of same size. `Resize` method is used to resize the image to specified size. Here it's `192`. While resizing, the image may be cropped/squished/anything else. Here, we are asking to `squish` the image.